# Basic solution - Exercise 1

Simple functions for the engineering constants of a unidirectional fibre composite. Use the same unit for every modulus (for example GPa).

In [1]:
def young_modulus_1(E_f, E_m, v_f):
    # E_f: Fibre Young's modulus [GPa]
    # E_m: Matrix Young's modulus [GPa]
    # v_f: Fibre volume fraction [-]
    return v_f * E_f + (1 - v_f) * E_m


def young_modulus_2(E_f, E_m, v_f, xi=2):
    # E_f: Fibre Young's modulus [GPa]
    # E_m: Matrix Young's modulus [GPa]
    # v_f: Fibre volume fraction [-]
    # xi: Halpin-Tsai geometry factor [-]
    eta = (E_f / E_m - 1) / (E_f / E_m + xi)
    return E_m * (1 + xi * eta * v_f) / (1 - eta * v_f)


def poissons_ratio_12(nu_f, nu_m, v_f):
    # nu_f: Fibre Poisson's ratio [-]
    # nu_m: Matrix Poisson's ratio [-]
    # v_f: Fibre volume fraction [-]
    return v_f * nu_f + (1 - v_f) * nu_m


def poissons_ratio_21(nu_12, E_1, E_2):
    # nu_12: Major Poisson's ratio [-]
    # E_1: Longitudinal Young's modulus [GPa]
    # E_2: Transverse Young's modulus [GPa]
    return nu_12 * E_2 / E_1

In [2]:
def constituent_bulk_modulus(E, nu):
    # E: Constituent Young's modulus [GPa]
    # nu: Constituent Poisson's ratio [-]
    return E / (3 * (1 - 2 * nu))


def bulk_modulus(E_f, nu_f, E_m, nu_m, v_f):
    # E_f: Fibre Young's modulus [GPa]
    # nu_f: Fibre Poisson's ratio [-]
    # E_m: Matrix Young's modulus [GPa]
    # nu_m: Matrix Poisson's ratio [-]
    # v_f: Fibre volume fraction [-]
    K_f = constituent_bulk_modulus(E_f, nu_f)
    K_m = constituent_bulk_modulus(E_m, nu_m)
    return 1 / (v_f / K_f + (1 - v_f) / K_m)


def poissons_ratio_23(nu_21, E_2, K):
    # nu_21: Minor Poisson's ratio [-]
    # E_2: Transverse Young's modulus [GPa]
    # K: Composite bulk modulus [GPa]
    return 1 - nu_21 - E_2 / (3 * K)


def constituent_shear_modulus(E, nu):
    # E: Constituent Young's modulus [GPa]
    # nu: Constituent Poisson's ratio [-]
    return E / (2 * (1 + nu))


def shear_modulus_12(E_f, nu_f, E_m, nu_m, v_f, xi=1):
    # E_f: Fibre Young's modulus [GPa]
    # nu_f: Fibre Poisson's ratio [-]
    # E_m: Matrix Young's modulus [GPa]
    # nu_m: Matrix Poisson's ratio [-]
    # v_f: Fibre volume fraction [-]
    # xi: Halpin-Tsai geometry factor [-]
    G_f = constituent_shear_modulus(E_f, nu_f)
    G_m = constituent_shear_modulus(E_m, nu_m)
    eta = (G_f / G_m - 1) / (G_f / G_m + xi)
    return G_m * (1 + xi * eta * v_f) / (1 - eta * v_f)


def shear_modulus_23(E_2, nu_23):
    # E_2: Transverse Young's modulus [GPa]
    # nu_23: Transverse Poisson's ratio [-]
    return E_2 / (2 * (1 + nu_23))


def density(rho_f, rho_m, v_f):
    # rho_f: Fibre density [any consistent density unit]
    # rho_m: Matrix density [same unit as rho_f]
    # v_f: Fibre volume fraction [-]
    return v_f * rho_f + (1 - v_f) * rho_m

In [3]:
def compliance_matrix_2d(E_1, E_2, nu_12, G_12):
    # E_1: Longitudinal Young's modulus [GPa]
    # E_2: Transverse Young's modulus [GPa]
    # nu_12: Major Poisson's ratio [-]
    # G_12: In-plane shear modulus [GPa]
    return [
        [1 / E_1, -nu_12 / E_1, 0],
        [-nu_12 / E_1, 1 / E_2, 0],
        [0, 0, 1 / G_12],
    ]


def stiffness_matrix_2d(E_1, E_2, nu_12, nu_21, G_12):
    # E_1: Longitudinal Young's modulus [GPa]
    # E_2: Transverse Young's modulus [GPa]
    # nu_12: Major Poisson's ratio [-]
    # nu_21: Minor Poisson's ratio [-]
    # G_12: In-plane shear modulus [GPa]
    d = 1 - nu_12 * nu_21
    Q_11 = E_1 / d
    Q_22 = E_2 / d
    Q_12 = nu_12 * E_2 / d
    return [
        [Q_11, Q_12, 0],
        [Q_12, Q_22, 0],
        [0, 0, G_12],
    ]


def compliance_matrix_3d(E_1, E_2, nu_12, nu_23, G_12, G_23):
    # E_1: Longitudinal Young's modulus [GPa]
    # E_2: Transverse Young's modulus [GPa]
    # nu_12: Major Poisson's ratio [-]
    # nu_23: Transverse Poisson's ratio [-]
    # G_12: In-plane shear modulus [GPa]
    # G_23: Transverse shear modulus [GPa]
    S = [[0] * 6 for _ in range(6)]
    S[0][0], S[1][1], S[2][2] = 1 / E_1, 1 / E_2, 1 / E_2
    S[0][1] = S[1][0] = S[0][2] = S[2][0] = -nu_12 / E_1
    S[1][2] = S[2][1] = -nu_23 / E_2
    S[3][3], S[4][4], S[5][5] = 1 / G_23, 1 / G_12, 1 / G_12
    return S


def print_matrix(matrix):
    # matrix: Two-dimensional list of numerical matrix entries
    for row in matrix:
        print('  '.join(f'{value:.3f}' for value in row))

## Example

The result from each function is used as input to the next function.

In [4]:
E_f, nu_f = 76, 0.20
E_m, nu_m = 4, 0.30
v_f = 0.55

E_1 = young_modulus_1(E_f, E_m, v_f)
E_2 = young_modulus_2(E_f, E_m, v_f)
nu_12 = poissons_ratio_12(nu_f, nu_m, v_f)
nu_21 = poissons_ratio_21(nu_12, E_1, E_2)
K = bulk_modulus(E_f, nu_f, E_m, nu_m, v_f)
nu_23 = poissons_ratio_23(nu_21, E_2, K)
G_12 = shear_modulus_12(E_f, nu_f, E_m, nu_m, v_f)
G_23 = shear_modulus_23(E_2, nu_23)

for name, value in [('E1', E_1), ('E2', E_2), ('nu12', nu_12),
                    ('nu21', nu_21), ('nu23', nu_23), ('K', K),
                    ('G12', G_12), ('G23', G_23)]:
    print(f'{name} = {value:.3f}')

E1 = 43.600
E2 = 14.703
nu12 = 0.245
nu21 = 0.083
nu23 = 0.192
K = 6.756
G12 = 4.604
G23 = 6.168


In [5]:
S = compliance_matrix_2d(E_1, E_2, nu_12, G_12)
Q = stiffness_matrix_2d(E_1, E_2, nu_12, nu_21, G_12)

print('S matrix:')
print_matrix(S)
print('\nQ matrix:')
print_matrix(Q)

S matrix:
0.023  -0.006  0.000
-0.006  0.068  0.000
0.000  0.000  0.217

Q matrix:
44.501  3.677  0.000
3.677  15.006  0.000
0.000  0.000  4.604
